# 03 - Automatic differentiation over tracking

The Enzyme extension computes one Jacobian per initial condition using the packed tracking path. Array layout is `(particle, output coordinate, input coordinate)`.

In [ ]:
import Pkg
EXAMPLES_DIR = isfile(joinpath(pwd(), "common.jl")) ? pwd() : joinpath(pwd(), "examples")
Pkg.activate(joinpath(EXAMPLES_DIR, "environments", "ad"))
Pkg.instantiate()
using TrackPad, Enzyme

include(joinpath(EXAMPLES_DIR, "common.jl")); using .TrackPadExamples

In [ ]:
ring, beam = madx_fodo()
gl = GPULattice(ring, beam; dtype=Float64)
coordinates = [
     8e-4  2e-4   6e-4  -1.5e-4  1e-4  -2e-4
    -5e-4 -1e-4   3e-4   1.0e-4 -2e-4   1e-4
]

In [ ]:
jacobian = zeros(Float64, size(coordinates, 1), 6, 6)
batch_jacobian!(jacobian, coordinates, gl)

jacobian[1, :, :]

In [ ]:
function tracked(input)
    output = copy(input)
    batch_linepass!(output, gl)
    return output
end

h = 1e-6
finite_difference = similar(jacobian)
for input_coordinate in 1:6
    plus, minus = copy(coordinates), copy(coordinates)
    plus[:, input_coordinate] .+= h
    minus[:, input_coordinate] .-= h
    finite_difference[:, :, input_coordinate] .=
        (tracked(plus) .- tracked(minus)) ./ (2h)
end

difference = jacobian .- finite_difference
max_abs_error = maximum(abs, difference)
max_scale = max(maximum(abs, jacobian), maximum(abs, finite_difference), eps())
(; agreement=isapprox(jacobian, finite_difference; rtol=2e-7, atol=2e-9),
   max_abs_error, max_rel_error=max_abs_error / max_scale)

The input array is not modified. `batch_hessian_vector_product!` is usually preferable to a full `batch_hessian!`; both use centered differences of Enzyme Jacobians because nested-Enzyme GPU Hessians are not currently supported.